# Part 3 results walkthrough

This notebook **reads the outputs produced by `python run_all.py`** (metrics tables, registry, monitoring report) and demonstrates the recommendation engine live. All modelling code lives in the Python modules so it can be tested, logged and reproduced; the notebook is a guided tour for reviewers.

> **Proxy label notice:** no accident dataset was available. `high_risk` is a documented proxy (High/Severe congestion during severe or low-visibility weather) and is not a prediction of real accidents.

In [1]:
import json, sys, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option('display.precision', 3)
METRICS = ROOT / 'reports' / 'metrics'

## Task 1 – Supervised models (test set, Jan–Sep 2018)

Classification of the proxy risk label, including the no-ML rule reference:

In [2]:
pd.read_csv(METRICS / 'classification_metrics.csv')[['candidate','test_accuracy','test_precision','test_recall','test_f1','test_roc_auc','test_pr_auc']]

,candidate,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,rule_reference_daytime_risky_weather,0.986,0.810,0.975,0.885,0.981,0.791
1,logistic_regression_v1,0.989,0.833,1.000,0.909,0.999,0.981
2,random_forest_v1,0.995,0.919,0.992,0.954,1.000,0.993
3,hist_gradient_boosting_v1,0.995,0.928,0.983,0.955,1.000,0.993
4,hist_gradient_boosting_v2,0.995,0.917,0.994,0.954,1.000,0.994


Traffic-volume regression:

In [3]:
pd.read_csv(METRICS / 'regression_metrics.csv')[['candidate','val_mae','test_mae','test_rmse','test_r2']]

,candidate,val_mae,test_mae,test_rmse,test_r2
0,ridge_regression_v1,569.986,550.073,709.008,0.871
1,random_forest_v1,231.759,218.271,369.455,0.965
2,hist_gradient_boosting_v1,231.543,217.317,363.875,0.966
3,hist_gradient_boosting_v2,240.116,221.498,366.766,0.966


![Regression metrics](../figures/task1_supervised/task1_regression_metrics.png)

## Task 2 – Unsupervised learning

In [4]:
pd.read_csv(METRICS / 'kmeans_cluster_profiles.csv')[['cluster','name','share_of_hours','mean_volume','weekend_share','adverse_weather_share']]

,cluster,name,share_of_hours,mean_volume,weekend_share,adverse_weather_share
0,0,"Low traffic, night, mixed days",0.322,1037.373,0.327,0.0
1,1,"Moderate traffic, all hours, mixed days, adver...",0.076,3005.799,0.278,1.0
2,2,"Heavy traffic, afternoon-evening, mixed days",0.281,4196.218,0.292,0.0
3,3,"Heavy traffic, morning-midday, mixed days",0.320,4839.151,0.241,0.0


In [5]:
for text in pd.read_csv(METRICS / 'association_rules_top_lift.csv')['explanation'].head(5):
    print('-', text)

- During the night (00–05) at weekends with freezing temperatures, congestion is Low 95% of the time, 3.8x the overall rate of 25% (based on 914 hours).
- During the morning peak (06–09) on weekdays with cold temperatures, congestion is Severe 92% of the time, 3.7x the overall rate of 25% (based on 942 hours).
- During the afternoon peak (15–18) at weekends with warm temperatures, congestion is High 92% of the time, 3.7x the overall rate of 25% (based on 670 hours).
- During the afternoon peak (15–18) at weekends in rain/drizzle weather, congestion is High 91% of the time, 3.6x the overall rate of 25% (based on 226 hours).
- During the night (00–05) at weekends in low visibility weather, congestion is Low 91% of the time, 3.6x the overall rate of 25% (based on 358 hours).


![Clusters](../figures/task2_unsupervised/task2_kmeans_clusters_hour_volume.png)

## Task 3 – LSTM vs baselines and SHAP

In [6]:
pd.read_csv(METRICS / 'deep_learning_comparison.csv')

,model,mae,rmse,r2
0,Persistence (last hour),588.292,813.487,0.830
1,Seasonal naive (same hour yesterday),566.073,1030.955,0.727
2,Task 1 regressor (no lags),217.113,363.286,0.966
3,Hist GB + lag features (surrogate),139.023,216.577,0.988
4,LSTM v1 small,156.163,233.199,0.986
5,LSTM v2 stacked,158.237,241.707,0.985


In [7]:
pd.read_csv(METRICS / 'shap_importance_next_hour_surrogate.csv').head(8)

,feature,mean_abs_shap,share
0,lag_1,1169.889,0.493
1,hour_cos,532.189,0.224
2,hour,186.777,0.079
3,is_rush_hour,136.293,0.057
4,lag_24,77.229,0.033
5,hour_sin,64.455,0.027
6,rolling_mean_24,45.512,0.019
7,lag_3,39.036,0.016


![SHAP beeswarm](../figures/task3_explainability/task3_shap_surrogate_beeswarm.png)

## Task 5 – Recommendation engine (live call)

In [8]:
from datetime import date
from recommender import TravelRecommender, TripRequest
rec = TravelRecommender().recommend(TripRequest(date(2018, 3, 13), earliest_hour=7, latest_hour=19, window_hours=1, weather='Snow'))
print(rec.message)

For a weekday journey on Tuesday 13 March 2018 in snow, consider travelling between 19:00 and 20:00, when traffic is forecast at about 2,942 vehicles per hour (Medium congestion). That is roughly 54% lighter than the busiest option in your range, 16:00-17:00 (~6,430). Historically this weekday slot averages about 3,361 vehicles per hour. Good alternatives: 10:00-11:00 (~4,207), 18:00-19:00 (~4,341). Snow lowers daytime volumes by about 8% historically but makes each journey slower and riskier; allow extra time.


## Task 6 – Model versions and monitoring

In [9]:
registry = json.loads((ROOT / 'models' / 'model_registry.json').read_text())
pd.DataFrame([{'model': m, 'version': v['version'], 'candidate': v['candidate'], 'champion': v['is_champion'], 'val_metric': v['validation'].get('mae', v['validation'].get('pr_auc'))} for m, b in registry.items() for v in b['versions']])

,model,version,candidate,champion,val_metric
0,traffic-risk-classifier,1,logistic_regression_v1,False,0.991
1,traffic-risk-classifier,2,random_forest_v1,False,0.995
2,traffic-risk-classifier,3,hist_gradient_boosting_v1,False,0.996
3,traffic-risk-classifier,4,hist_gradient_boosting_v2,True,0.997
4,traffic-volume-regressor,1,ridge_regression_v1,False,569.986
5,traffic-volume-regressor,2,random_forest_v1,False,231.759
6,traffic-volume-regressor,3,hist_gradient_boosting_v1,True,231.543
7,traffic-volume-regressor,4,hist_gradient_boosting_v2,False,240.116
8,traffic-volume-lstm,1,lstm_v1_small,True,168.520
9,traffic-volume-lstm,2,lstm_v2_stacked,False,169.282


In [10]:
report = json.loads((ROOT / 'monitoring' / 'monitoring_report.json').read_text())
print('SYSTEM STATUS:', report['system_status'], '| alerts during period:', report['alerts_in_period'])
pd.DataFrame([{k: b[k] for k in ('batch','kind','status','mae','mae_ratio')} | {'alerts': '; '.join(b['alerts'])} for b in report['batches']])

SYSTEM STATUS: PASS | alerts during period: ['2018-01', '2018-04']


,batch,kind,status,mae,mae_ratio,alerts
0,2018-01,real,ALERT,307.051,1.326,Error drift: MAE 307 is 1.33x the validation M...
1,2018-02,real,PASS,247.231,1.068,
2,2018-03,real,PASS,217.763,0.940,
3,2018-04,real,ALERT,302.871,1.308,Error drift: MAE 303 is 1.31x the validation M...
4,2018-05,real,PASS,185.946,0.803,
5,2018-06,real,PASS,167.239,0.722,
6,2018-07,real,PASS,181.575,0.784,
7,2018-08,real,PASS,167.749,0.724,
8,2018-09,real,PASS,181.169,0.782,
9,SIM: sensor under-count (Sep 2018),simulated,ALERT,1179.427,5.094,"Error drift: MAE 1,179 is 5.09x the validation..."


## Task 7 – Uneven errors

In [11]:
seg = pd.read_csv(METRICS / 'fairness_regression_errors_by_segment.csv')
seg.sort_values('mae_vs_overall', ascending=False).head(6)

,segment_type,segment,hours,mae,relative_mae,bias,mae_vs_overall
12,weather_group,Snow,453,588.383,0.211,333.932,2.708
5,day_type,Holiday,167,388.766,0.159,182.430,1.789
4,hour_band,PM peak 15-18,1092,308.163,0.060,21.509,1.418
17,season,Winter,1414,278.622,0.087,6.819,1.282
7,day_type,Weekend,1862,262.805,0.099,13.052,1.209
1,hour_band,Evening 19-23,1365,258.353,0.101,9.231,1.189


See `reports/BIAS_FAIRNESS_REPORT.md` and `reports/FINAL_CAPSTONE_REPORT.md` for the full discussion.